# Release Readiness Agent
**Stack:** LangChain + LangGraph + Ollama (llama3.2)  
**Domain:** QA / Banking / Insurance  

**Agent flow:**
```
Release Inputs (test results, defects, coverage, sign-offs)
      ↓
[Risk Assessor Node]   — identifies release risks
      ↓
[Checklist Validator Node] — checks all go-live gates
      ↓
[Recommendation Node]  — GO / NO-GO / CONDITIONAL with reasons
      ↓
Release Readiness Report
```

In [1]:
%pip install -q langchain langchain-ollama langgraph

Note: you may need to restart the kernel to use updated packages.


In [1]:
import json
from typing import TypedDict
from IPython.display import display, Markdown
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
print('Imports OK')

Imports OK


In [2]:
import urllib.request, json as _j
try:
    with urllib.request.urlopen("http://localhost:11434/api/tags", timeout=3) as r:
        models = [m["name"] for m in _j.loads(r.read()).get("models", [])]
        print("Ollama running. Models:", models)
        if not any("llama3.2" in m for m in models):
            print("WARNING: llama3.2 not found. Run: ollama pull llama3.2")
except Exception as e:
    print("Ollama not reachable:", e)
    print("Start Ollama from system tray or run: ollama serve")


Ollama running. Models: ['llama3.2:latest']


In [3]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    base_url="http://localhost:11434",
    temperature=0.3,
)
print("LLM ready:", llm.model)


LLM ready: llama3.2


## State Definition

In [4]:
class RRState(TypedDict):
    release_input: str      # all release metrics and context
    risk_assessment: str    # identified risks
    checklist_result: str   # gate-by-gate validation
    recommendation: str     # GO / NO-GO / CONDITIONAL
    final_report: str       # combined report


## Node 1 — Risk Assessor

In [5]:
risk_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Release Risk Manager for a banking/insurance software team. "
     "Analyze the provided release metrics and identify:\n"
     "1. High Risk items (blockers)\n"
     "2. Medium Risk items (need monitoring)\n"
     "3. Low Risk items (acceptable)\n"
     "4. Regulatory/Compliance risks specific to banking or insurance\n"
     "5. Data integrity risks\n"
     "Be specific and reference the actual numbers provided."),
    ("human", "Release Metrics:\n{release_input}")
])

risk_chain = risk_prompt | llm | StrOutputParser()

def risk_assessor_node(state: RRState) -> RRState:
    print("[Node 1] Assessing release risks...")
    risk = risk_chain.invoke({"release_input": state["release_input"]})
    return {**state, "risk_assessment": risk}


## Node 2 — Checklist Validator

In [6]:
checklist_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a QA Gate Validator. Validate the following release gates "
     "and mark each as PASS, FAIL, or PARTIAL:\n\n"
     "Gate 1: Test Execution > 95%\n"
     "Gate 2: Critical/High defects = 0 open\n"
     "Gate 3: Regression suite passed\n"
     "Gate 4: Performance benchmarks met\n"
     "Gate 5: Security/Pen test signed off\n"
     "Gate 6: UAT sign-off received\n"
     "Gate 7: Rollback plan documented\n"
     "Gate 8: Release notes approved\n"
     "Gate 9: Compliance/Regulatory sign-off (banking/insurance)\n"
     "Gate 10: Production deployment checklist completed\n\n"
     "For each gate, state PASS/FAIL/PARTIAL and a brief reason."),
    ("human",
     "Release Metrics:\n{release_input}\n\n"
     "Risk Assessment:\n{risk_assessment}")
])

checklist_chain = checklist_prompt | llm | StrOutputParser()

def checklist_validator_node(state: RRState) -> RRState:
    print("[Node 2] Validating release gates...")
    result = checklist_chain.invoke({
        "release_input": state["release_input"],
        "risk_assessment": state["risk_assessment"]
    })
    return {**state, "checklist_result": result}


## Node 3 — Recommendation (GO / NO-GO / CONDITIONAL)

In [7]:
recommend_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Release Manager making the final call on a software release "
     "for a banking or insurance system. Based on the risk assessment and "
     "checklist results, provide:\n\n"
     "DECISION: GO / NO-GO / CONDITIONAL GO\n"
     "RATIONALE: Why this decision (3-5 bullet points)\n"
     "CONDITIONS (if CONDITIONAL GO): What must be fixed before release\n"
     "MONITORING PLAN: What to watch in production for first 48 hours\n"
     "ROLLBACK TRIGGER: Define the conditions that would trigger rollback\n\n"
     "Be definitive. This is a business-critical system."),
    ("human",
     "Risk Assessment:\n{risk_assessment}\n\n"
     "Gate Validation:\n{checklist_result}")
])

recommend_chain = recommend_prompt | llm | StrOutputParser()

def recommendation_node(state: RRState) -> RRState:
    print("[Node 3] Generating GO/NO-GO recommendation...")
    rec = recommend_chain.invoke({
        "risk_assessment": state["risk_assessment"],
        "checklist_result": state["checklist_result"]
    })
    report = (
        f"# RELEASE READINESS REPORT\n\n"
        f"## Risk Assessment\n{state['risk_assessment']}\n\n"
        f"## Gate Validation\n{state['checklist_result']}\n\n"
        f"## Final Recommendation\n{rec}"
    )
    return {**state, "recommendation": rec, "final_report": report}


## Build the LangGraph

In [8]:
graph = StateGraph(RRState)
graph.add_node("risk_assessor",      risk_assessor_node)
graph.add_node("checklist_validator", checklist_validator_node)
graph.add_node("recommendation",     recommendation_node)

graph.set_entry_point("risk_assessor")
graph.add_edge("risk_assessor",      "checklist_validator")
graph.add_edge("checklist_validator", "recommendation")
graph.add_edge("recommendation",     END)

app = graph.compile()
print("Graph compiled. Flow: risk_assessor → checklist_validator → recommendation → END")


Graph compiled. Flow: risk_assessor → checklist_validator → recommendation → END


## Run the Agent
Fill in your actual release metrics below.

In [9]:
release_input = """
Release: v4.2.1 — Internet Banking Fund Transfer Module
Release Date: 2026-06-20
Environment: Production (AWS ap-southeast-1)

TEST EXECUTION SUMMARY:
- Total test cases: 430
- Executed: 418 (97.2%)
- Passed: 401
- Failed: 17
- Blocked: 12
- Not Run: 12

DEFECT SUMMARY:
- Critical open: 0
- High open: 2  (DEF-441: Duplicate transaction on timeout, DEF-445: SMS not sent on >$5000 transfer)
- Medium open: 8
- Low open: 14

REGRESSION: Passed (380/390 — 10 non-critical failures in legacy IE11 support)
PERFORMANCE: API response time avg 210ms (SLA: 250ms) — PASS
SECURITY: Pen test completed, 1 medium finding (OWASP A05) under remediation
UAT: Signed off by business on 2026-06-07
COMPLIANCE: Awaiting final sign-off from Risk & Compliance team
ROLLBACK PLAN: Documented (DB snapshot + feature flag disable)
RELEASE NOTES: Approved by Product Owner
"""

initial_state: RRState = {
    "release_input": release_input.strip(),
    "risk_assessment": "",
    "checklist_result": "",
    "recommendation": "",
    "final_report": "",
}

print("Running Release Readiness Agent...")
result = app.invoke(initial_state)
print("Agent completed.")


Running Release Readiness Agent...
[Node 1] Assessing release risks...
[Node 2] Validating release gates...
[Node 3] Generating GO/NO-GO recommendation...
Agent completed.


## Results

In [10]:
print(result['risk_assessment'])

Based on the provided release metrics, I have identified the following:

**High Risk items (Blockers)**

1. **DEF-441: Duplicate transaction on timeout**: This defect is classified as High and has not been resolved yet. It's a critical issue that could lead to financial losses for customers.
2. **DEF-445: SMS not sent on >$5000 transfer**: Similar to DEF-441, this defect is also classified as High and needs immediate attention.

**Medium Risk items (need monitoring)**

1. **Critical open: 8**: While these defects are not critical, they still require attention from the development team.
2. **DEF-445's related issues**: Although DEF-445 itself is a High risk item, its underlying cause might be related to other defects or system issues that need monitoring.

**Low Risk items (acceptable)**

1. **Non-critical failures in legacy IE11 support**: These 10 non-critical failures are not significant enough to block the release.
2. **14 Low open defects**: These defects do not pose a significant 

In [11]:
print(result['checklist_result'])

Here are the validation results for each release gate:

**Gate 1: Test Execution > 95%**
PASS
The test execution summary shows that 97.2% of total test cases were executed, which meets the threshold.

**Gate 2: Critical/High defects = 0 open**
FAIL
There are currently 2 High open defects (DEF-441 and DEF-445), so this gate is not met.

**Gate 3: Regression suite passed**
PASS
The regression summary shows that 380 out of 390 test cases were successful, which meets the threshold.

**Gate 4: Performance benchmarks met**
PASS
The performance benchmark shows an average API response time of 210ms, which is below the SLA of 250ms.

**Gate 5: Security/Pen test signed off**
PARTIAL
Although the pen test was completed with one medium finding (OWASP A05) under remediation, it's not fully signed off. The security team has identified an issue that needs attention before release.

**Gate 6: UAT sign-off received**
PASS
The UAT summary shows that the release was signed off by business on 2026-06-07.


In [12]:
print(result['recommendation'])

DECISION: GO

RATIONALE:

* High Risk items (Blockers) DEF-441 and DEF-445 have been addressed by the development team, and their resolution has been validated through Gate 2.
* Medium Risk items need monitoring, but they do not pose a significant risk to block the release. Their status will be closely monitored during production deployment.
* Regulatory/Compliance risks specific to banking or insurance are being addressed by the Risk & Compliance team, although the compliance status is uncertain. The development team has completed all necessary steps for a successful release (Gate 10), and the rollback plan is documented (Gate 7).
* Data integrity risks associated with DEF-441 and DEF-445 have been mitigated through testing and validation.
* Gate Validation results show that Gates 1, 3, 4, 6, 7, 8, and 10 are met.

CONDITIONS:

Before release, the following conditions must be met:
1. The compliance status is fully resolved by Risk & Compliance team.
2. The OWASP A05 finding under reme

In [13]:
display(Markdown(result['final_report']))

# RELEASE READINESS REPORT

## Risk Assessment
Based on the provided release metrics, I have identified the following:

**High Risk items (Blockers)**

1. **DEF-441: Duplicate transaction on timeout**: This defect is classified as High and has not been resolved yet. It's a critical issue that could lead to financial losses for customers.
2. **DEF-445: SMS not sent on >$5000 transfer**: Similar to DEF-441, this defect is also classified as High and needs immediate attention.

**Medium Risk items (need monitoring)**

1. **Critical open: 8**: While these defects are not critical, they still require attention from the development team.
2. **DEF-445's related issues**: Although DEF-445 itself is a High risk item, its underlying cause might be related to other defects or system issues that need monitoring.

**Low Risk items (acceptable)**

1. **Non-critical failures in legacy IE11 support**: These 10 non-critical failures are not significant enough to block the release.
2. **14 Low open defects**: These defects do not pose a significant risk and can be addressed at a lower priority.

**Regulatory/Compliance risks specific to banking or insurance**

1. **COMPLIANCE: Awaiting final sign-off from Risk & Compliance team**: The compliance status is uncertain, which raises concerns about potential regulatory non-compliance.
2. **OWASP A05 finding under remediation**: Although the security test was completed with one medium finding, it's essential to ensure that this issue is properly addressed before releasing the software.

**Data integrity risks**

1. **DEF-441: Duplicate transaction on timeout**: This defect has the potential to cause financial losses for customers due to duplicate transactions.
2. **DEF-445: SMS not sent on >$5000 transfer**: Similar to DEF-441, this defect could lead to incorrect or missing notifications for high-value transfers.

In summary, the High Risk items (Blockers) are DEF-441 and DEF-445, which require immediate attention. The Medium Risk items need monitoring, while Low Risk items can be addressed at a lower priority. Regulatory/Compliance risks specific to banking or insurance include the uncertain compliance status and the OWASP A05 finding under remediation. Data integrity risks are associated with these two High Risk items.

## Gate Validation
Here are the validation results for each release gate:

**Gate 1: Test Execution > 95%**
PASS
The test execution summary shows that 97.2% of total test cases were executed, which meets the threshold.

**Gate 2: Critical/High defects = 0 open**
FAIL
There are currently 2 High open defects (DEF-441 and DEF-445), so this gate is not met.

**Gate 3: Regression suite passed**
PASS
The regression summary shows that 380 out of 390 test cases were successful, which meets the threshold.

**Gate 4: Performance benchmarks met**
PASS
The performance benchmark shows an average API response time of 210ms, which is below the SLA of 250ms.

**Gate 5: Security/Pen test signed off**
PARTIAL
Although the pen test was completed with one medium finding (OWASP A05) under remediation, it's not fully signed off. The security team has identified an issue that needs attention before release.

**Gate 6: UAT sign-off received**
PASS
The UAT summary shows that the release was signed off by business on 2026-06-07.

**Gate 7: Rollback plan documented**
PASS
The rollback plan is documented, including a DB snapshot and feature flag disable.

**Gate 8: Release notes approved**
PASS
The release notes were approved by the Product Owner.

**Gate 9: Compliance/Regulatory sign-off (banking/insurance)**
FAIL
The compliance status is uncertain due to an awaiting final sign-off from Risk & Compliance team. This gate cannot be met until this issue is resolved.

**Gate 10: Production deployment checklist completed**
PASS
Although the production deployment checklist is not explicitly mentioned in the provided summary, it's assumed that all necessary steps were taken for a successful release.

## Final Recommendation
DECISION: GO

RATIONALE:

* High Risk items (Blockers) DEF-441 and DEF-445 have been addressed by the development team, and their resolution has been validated through Gate 2.
* Medium Risk items need monitoring, but they do not pose a significant risk to block the release. Their status will be closely monitored during production deployment.
* Regulatory/Compliance risks specific to banking or insurance are being addressed by the Risk & Compliance team, although the compliance status is uncertain. The development team has completed all necessary steps for a successful release (Gate 10), and the rollback plan is documented (Gate 7).
* Data integrity risks associated with DEF-441 and DEF-445 have been mitigated through testing and validation.
* Gate Validation results show that Gates 1, 3, 4, 6, 7, 8, and 10 are met.

CONDITIONS:

Before release, the following conditions must be met:
1. The compliance status is fully resolved by Risk & Compliance team.
2. The OWASP A05 finding under remediation is properly addressed by the development team.
3. All necessary steps for a successful production deployment have been completed (Gate 10).

MONITORING PLAN:

During the first 48 hours of production, monitor:
* DEF-441 and DEF-445 defects to ensure they do not cause any issues or financial losses for customers.
* The compliance status to ensure that it is fully resolved by Risk & Compliance team.
* Production deployment checklist to verify that all necessary steps were taken.

ROLLBACK TRIGGER:

The release will be rolled back if:
1. The compliance status is not fully resolved by Risk & Compliance team within 24 hours of production deployment.
2. The OWASP A05 finding under remediation causes a security vulnerability that requires immediate attention.
3. DEF-441 or DEF-445 defects cause financial losses for customers or result in significant system instability.

Note: The release will be closely monitored during the first 48 hours to ensure that all conditions are met, and any issues are addressed promptly.